In [2]:
import pandas as pd

In [3]:
df_train = pd.read_parquet("../../data/churn-prediction-25-26/train.parquet")

In [24]:
"""
Computes the total number of sessions per user

mean time per session per user

mean days between each session per user

days since last session per user
"""

df = df_train.copy()

df = df[["userId", "sessionId", "time"]]

df["session_time_end"] = df.groupby(["userId", "sessionId"])["time"].transform("max")

In [25]:
df

,userId,sessionId,time,session_time_end
0,1749042,22683,2018-10-01 00:00:01,2018-10-01 03:10:06
992,1749042,22683,2018-10-01 00:08:45,2018-10-01 03:10:06
1360,1749042,22683,2018-10-01 00:11:43,2018-10-01 03:10:06
1825,1749042,22683,2018-10-01 00:15:35,2018-10-01 03:10:06
2366,1749042,22683,2018-10-01 00:20:00,2018-10-01 03:10:06
...,...,...,...,...
25661184,1152881,2537,2018-11-19 23:50:04,2018-11-19 23:58:33
25661214,1152881,2537,2018-11-19 23:50:43,2018-11-19 23:58:33
25661320,1152881,2537,2018-11-19 23:53:07,2018-11-19 23:58:33
25661462,1152881,2537,2018-11-19 23:55:50,2018-11-19 23:58:33


In [26]:
df["session_time_start"] = df.groupby(["userId", "sessionId"])["time"].\
    transform("min")

In [27]:
df["time_per_session"] = df["session_time_end"] - df["session_time_start"]

In [28]:
df.drop(columns="time", inplace=True)
df.drop_duplicates(inplace=True)

In [29]:
df

,userId,sessionId,session_time_end,session_time_start,time_per_session
0,1749042,22683,2018-10-01 03:10:06,2018-10-01 00:00:01,0 days 03:10:05
388853,1749042,25570,2018-10-02 08:00:26,2018-10-02 07:46:53,0 days 00:13:33
879957,1749042,31304,2018-10-04 09:23:21,2018-10-03 18:22:52,0 days 15:00:29
1514872,1749042,41406,2018-10-06 02:23:56,2018-10-05 12:50:55,0 days 13:33:01
2345163,1749042,49702,2018-10-08 16:06:47,2018-10-08 15:51:24,0 days 00:15:23
...,...,...,...,...,...
25646375,1494594,778,2018-11-19 19:52:00,2018-11-19 19:02:23,0 days 00:49:37
25646627,1036641,70,2018-11-19 23:58:21,2018-11-19 19:06:31,0 days 04:51:50
25650010,1110980,2579,2018-11-19 23:55:01,2018-11-19 20:04:51,0 days 03:50:10
25650509,1594272,379,2018-11-19 23:48:58,2018-11-19 20:13:14,0 days 03:35:44


In [30]:
df = df.sort_values(["userId", "session_time_start"], ascending=True)

In [32]:
# time delta to previous session of the same useru
df["time_since_prev_session"] = (
    df.groupby("userId")["session_time_start"].diff()  # Timedelta
)

In [33]:
df

,userId,sessionId,session_time_end,session_time_start,time_per_session,time_since_prev_session
403570,1000025,23706,2018-10-02 10:05:09,2018-10-02 08:59:29,0 days 01:05:40,NaT
541288,1000025,31688,2018-10-03 21:31:44,2018-10-02 18:12:22,1 days 03:19:22,0 days 09:12:53
985117,1000025,39243,2018-10-04 14:42:06,2018-10-04 01:04:35,0 days 13:37:31,1 days 06:52:13
1366975,1000025,42490,2018-10-05 03:09:11,2018-10-05 01:36:46,0 days 01:32:25,1 days 00:32:11
1909331,1000025,45191,2018-10-06 22:19:23,2018-10-06 22:09:33,0 days 00:09:50,1 days 20:32:47
...,...,...,...,...,...,...
8625349,1999905,126577,2018-10-26 15:58:09,2018-10-26 15:25:08,0 days 00:33:01,0 days 08:33:17
10244265,1999905,127830,2018-10-31 18:23:50,2018-10-31 15:37:27,0 days 02:46:23,5 days 00:12:19
13664639,1999905,161072,2018-11-10 23:09:08,2018-11-10 23:09:06,0 days 00:00:02,10 days 07:31:39
15328695,1999905,179401,2018-11-15 22:51:35,2018-11-15 22:32:18,0 days 00:19:17,4 days 23:23:12


In [34]:
# mean time between sessions
df["mean_time_per_session"] = (
    df.groupby("userId")["time_per_session"].transform("mean")
)

df["median_time_per_session"] = (
    df.groupby("userId")["time_per_session"].transform("median")
)

In [35]:
df

,userId,sessionId,session_time_end,session_time_start,time_per_session,time_since_prev_session,mean_time_per_session,median_time_per_session
403570,1000025,23706,2018-10-02 10:05:09,2018-10-02 08:59:29,0 days 01:05:40,NaT,0 days 06:44:47.588235,0 days 05:41:48
541288,1000025,31688,2018-10-03 21:31:44,2018-10-02 18:12:22,1 days 03:19:22,0 days 09:12:53,0 days 06:44:47.588235,0 days 05:41:48
985117,1000025,39243,2018-10-04 14:42:06,2018-10-04 01:04:35,0 days 13:37:31,1 days 06:52:13,0 days 06:44:47.588235,0 days 05:41:48
1366975,1000025,42490,2018-10-05 03:09:11,2018-10-05 01:36:46,0 days 01:32:25,1 days 00:32:11,0 days 06:44:47.588235,0 days 05:41:48
1909331,1000025,45191,2018-10-06 22:19:23,2018-10-06 22:09:33,0 days 00:09:50,1 days 20:32:47,0 days 06:44:47.588235,0 days 05:41:48
...,...,...,...,...,...,...,...,...
8625349,1999905,126577,2018-10-26 15:58:09,2018-10-26 15:25:08,0 days 00:33:01,0 days 08:33:17,0 days 01:21:40,0 days 00:50:12
10244265,1999905,127830,2018-10-31 18:23:50,2018-10-31 15:37:27,0 days 02:46:23,5 days 00:12:19,0 days 01:21:40,0 days 00:50:12
13664639,1999905,161072,2018-11-10 23:09:08,2018-11-10 23:09:06,0 days 00:00:02,10 days 07:31:39,0 days 01:21:40,0 days 00:50:12
15328695,1999905,179401,2018-11-15 22:51:35,2018-11-15 22:32:18,0 days 00:19:17,4 days 23:23:12,0 days 01:21:40,0 days 00:50:12


In [36]:
# mean days between sessions
df["mean_time_between_sessions"] = (
    df.groupby("userId")["time_since_prev_session"].transform("mean")
)

df["median_time_between_sessions"] = (
    df.groupby("userId")["time_since_prev_session"].transform("median")
)

In [37]:
df

,userId,sessionId,session_time_end,session_time_start,time_per_session,time_since_prev_session,mean_time_per_session,median_time_per_session,mean_time_between_sessions,median_time_between_sessions
403570,1000025,23706,2018-10-02 10:05:09,2018-10-02 08:59:29,0 days 01:05:40,NaT,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04
541288,1000025,31688,2018-10-03 21:31:44,2018-10-02 18:12:22,1 days 03:19:22,0 days 09:12:53,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04
985117,1000025,39243,2018-10-04 14:42:06,2018-10-04 01:04:35,0 days 13:37:31,1 days 06:52:13,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04
1366975,1000025,42490,2018-10-05 03:09:11,2018-10-05 01:36:46,0 days 01:32:25,1 days 00:32:11,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04
1909331,1000025,45191,2018-10-06 22:19:23,2018-10-06 22:09:33,0 days 00:09:50,1 days 20:32:47,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04
...,...,...,...,...,...,...,...,...,...,...
8625349,1999905,126577,2018-10-26 15:58:09,2018-10-26 15:25:08,0 days 00:33:01,0 days 08:33:17,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43
10244265,1999905,127830,2018-10-31 18:23:50,2018-10-31 15:37:27,0 days 02:46:23,5 days 00:12:19,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43
13664639,1999905,161072,2018-11-10 23:09:08,2018-11-10 23:09:06,0 days 00:00:02,10 days 07:31:39,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43
15328695,1999905,179401,2018-11-15 22:51:35,2018-11-15 22:32:18,0 days 00:19:17,4 days 23:23:12,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43


In [38]:
#df["time_not_used"] = df["time_not_used"].dt.seconds // 60
df["mean_tps_minutes"] = df["mean_time_per_session"].dt.total_seconds() // 60
df["mean_tbs_minutes"] = df["mean_time_between_sessions"].dt.total_seconds() // 60
df["median_tps_minutes"] = df["median_time_per_session"].dt.total_seconds() // 60
df["median_tbs_minutes"] = df["median_time_between_sessions"].dt.total_seconds() // 60
#df["mean_days_between_sessions_miuntes"] = df["mean_days_between_sessions"].fillna(1000)

In [39]:
df.columns

Index(['userId', 'sessionId', 'session_time_end', 'session_time_start',
       'time_per_session', 'time_since_prev_session', 'mean_time_per_session',
       'median_time_per_session', 'mean_time_between_sessions',
       'median_time_between_sessions', 'mean_tps_minutes', 'mean_tbs_minutes',
       'median_tps_minutes', 'median_tbs_minutes'],
      dtype='object')

In [40]:
df

,userId,sessionId,session_time_end,session_time_start,time_per_session,time_since_prev_session,mean_time_per_session,median_time_per_session,mean_time_between_sessions,median_time_between_sessions,mean_tps_minutes,mean_tbs_minutes,median_tps_minutes,median_tbs_minutes
403570,1000025,23706,2018-10-02 10:05:09,2018-10-02 08:59:29,0 days 01:05:40,NaT,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04,404.0,1457.0,341.0,1078.0
541288,1000025,31688,2018-10-03 21:31:44,2018-10-02 18:12:22,1 days 03:19:22,0 days 09:12:53,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04,404.0,1457.0,341.0,1078.0
985117,1000025,39243,2018-10-04 14:42:06,2018-10-04 01:04:35,0 days 13:37:31,1 days 06:52:13,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04,404.0,1457.0,341.0,1078.0
1366975,1000025,42490,2018-10-05 03:09:11,2018-10-05 01:36:46,0 days 01:32:25,1 days 00:32:11,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04,404.0,1457.0,341.0,1078.0
1909331,1000025,45191,2018-10-06 22:19:23,2018-10-06 22:09:33,0 days 00:09:50,1 days 20:32:47,0 days 06:44:47.588235,0 days 05:41:48,1 days 00:17:02.437500,0 days 17:58:04,404.0,1457.0,341.0,1078.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8625349,1999905,126577,2018-10-26 15:58:09,2018-10-26 15:25:08,0 days 00:33:01,0 days 08:33:17,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43,81.0,5210.0,50.0,4077.0
10244265,1999905,127830,2018-10-31 18:23:50,2018-10-31 15:37:27,0 days 02:46:23,5 days 00:12:19,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43,81.0,5210.0,50.0,4077.0
13664639,1999905,161072,2018-11-10 23:09:08,2018-11-10 23:09:06,0 days 00:00:02,10 days 07:31:39,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43,81.0,5210.0,50.0,4077.0
15328695,1999905,179401,2018-11-15 22:51:35,2018-11-15 22:32:18,0 days 00:19:17,4 days 23:23:12,0 days 01:21:40,0 days 00:50:12,3 days 14:50:27.833333,2 days 19:57:43,81.0,5210.0,50.0,4077.0
